# **PROCEDIMIENTO COMPLETO DE CLASIFICACIÓN**

#01 Lectura de datos

In [ ]:
!pip install imbalanced-learn
!pip install xgboost


In [ ]:
# Cómo importar las librerías
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path_base = '/content/drive/MyDrive/Colab Notebooks'
dataset= pd.read_csv(f'{path_base}/Data_Cloro_Residual.csv', sep = ',')
display(dataset.head(3))
display(dataset.tail(3))

#02 Transformaciones de datos

## Analizar si están correlacionadas

In [ ]:
# Filtrar solo columnas numéricas
dataf_numeric = dataset.select_dtypes(include='number')

# Calcular matriz de correlación
corr_matrix = dataf_numeric.corr()

# Visualización con heatmap y barra ajustada
plt.figure(figsize=(14, 10))

ax = sns.heatmap(corr_matrix,
                 annot=True,
                 cmap='coolwarm',
                 fmt=".1f",
                 square=True,
                 annot_kws={'size': 6},
                 cbar=True,
                 cbar_kws={
                     'shrink': 0.8,           # Tamaño de la barra (0-1)
                     'aspect': 20,             # Relación ancho/alto
                     'pad': 0.02,              # Espacio entre heatmap y barra
                     'label': 'Correlación',   # Título de la barra
                     'orientation': 'vertical', # Orientación
                     'ticks': [-1, -0.5, 0, 0.5, 1]  # Valores específicos en la barra
                 })

# Ajustar etiquetas
plt.xticks(fontsize=10, rotation=45, ha='right')
plt.yticks(fontsize=10)

# Ajustar título
plt.title('Matriz de Correlación', fontsize=14, pad=20)

# Ajustar barra de color adicionalmente
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=9)  # Tamaño de números en la barra
cbar.set_label('Correlación', fontsize=10)  # Etiqueta de la barra

plt.tight_layout()
plt.show()

**En este punto, considerar:**

-Corrección de errores en la data categóricas

-Reagrupación de las categorias pocos frecuentes. Esta transformación se le aplicará a la data para despliegue

-Exploración de la data numérica

-Análisis de los outliers

-Selección de variables

In [ ]:
import os
import math as mt
import sklearn.model_selection as model_selection
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.compose import make_column_selector
from sklearn.naive_bayes import ComplementNB
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.base import TransformerMixin

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score
from imblearn.combine import SMOTETomek
import joblib

In [ ]:
# Obtén el 30% de los datos como una muestra aleatoria
muestra = dataset.sample(frac=0.9, random_state=42).reset_index(drop=True)

In [ ]:
muestra

###datosfinal para Train y Test

In [ ]:
X_train, X_test, y_train, y_test= model_selection.train_test_split(
    muestra.drop('Cloro_Adecuado', axis = 1),           ## TARGET
    muestra['Cloro_Adecuado'],                                  ## TARGET
    test_size=0.2,
    random_state=123,
    stratify= muestra['Cloro_Adecuado'])                         ## TARGET

In [ ]:
X_train.head(3)

## Pre Procesamiento de datos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score
from imblearn.combine import SMOTETomek
import joblib

# Dividir las variables en categóricas y numéricas usando X_train para asegurar consistencia
categorical_columns = X_train.select_dtypes(include='object').columns
numerical_columns = X_train.select_dtypes(include='number').columns

prep = ColumnTransformer([('encoder', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
                           ('scaler', StandardScaler(),numerical_columns),],
                          remainder= 'drop')

##**Otras opciones para num y cat**

**from sklearn.preprocessing import MinMaxScaler**

**('num', MinMaxScaler(), numeric_features)**   

Escala los datos numéricos en un rango específico (por defecto, entre 0 y 1).
Útil cuando las características numéricas tienen distribuciones no normales

-------------------------------------------------------------------------------

**from sklearn.preprocessing import RobustScaler**

**('num', RobustScaler(), numeric_features)**

Escala los datos utilizando estadísticas robustas para manejar outliers. Útil cuando tus datos numéricos contienen valores atípicos.

------------------------------------------------------------------------------
**from sklearn.preprocessing import OrdinalEncoder**

**('cat', OrdinalEncoder(), categorical_features)**

Útil cuando las variables categóricas tienen un orden intrínseco.Útil cuando las variables categóricas tienen un orden intrínseco. Revisar: https://bait509-ubc.github.io/BAIT509/lectures/lecture5.html        

## Opciones para preprocessors

In [ ]:
# Crear el preprocesador para no estandarizar datos
preprocessor2 = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_columns ),  # 'passthrough' indica que no se aplica ninguna transformación a las numéricas
        ('cat', OneHotEncoder(), categorical_columns )
    ])

In [ ]:
preprocessor3 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns),
        ('cat', OneHotEncoder(), categorical_columns)
    ],
    remainder='passthrough'
)

# as columnas que no son 'num' ni 'cat' se conservarán en el conjunto de datos preprocesado.
#  Esto puede ser útil si tienes características adicionales que no requieren transformación.

##CV

In [ ]:
cv= RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=123)
cv

##Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from imblearn.pipeline import Pipeline as Pipeline_imb
from imblearn.over_sampling import RandomOverSampler
from sklearn.linear_model import LogisticRegression

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as Pipeline_imb

#G Gradient Boosting

In [ ]:
from xgboost import XGBClassifier
import imblearn
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as Pipeline_imb
from imblearn.over_sampling import RandomOverSampler

In [ ]:
xg_pipe = Pipeline_imb([
                            ('preproc', prep),
                            ('upsamp', SMOTETomek(sampling_strategy=0.6)
                            #('upsamp', RandomOverSampler(sampling_strategy=0.5, random_state= 123) ),
                              #('upsamp', RandomUnderSampler(sampling_strategy=1) ),  #Balanceo, Under no da los mejores resultados
                             ),  #1 estan misma proporcionas las clases de target
                            ('xgboost', XGBClassifier (random_state=123)  )
                        ])

In [ ]:
xg_pipe.get_params().keys()

In [ ]:
xg_grid = dict(
    xgboost__max_depth= [2, 3, 5, 7, 10],
    xgboost__n_estimators= [10, 100, 500],
)

In [ ]:
#xg_grid = {
   # 'xgboost__max_depth': [2, 3, 5, 7, 10],
   # 'xgboost__n_estimators': [10, 100, 500],
   # 'xgboost__learning_rate': [0.01, 0.1, 0.2]  ,  # Tasa de aprendizaje, ajusta según sea necesario
   # 'xgboost__subsample': [0.8, 0.9, 1.0]  ,  # Fracción de muestras utilizadas para ajustar cada árbol
   # 'xgboost__colsample_bytree': [0.8, 1.0]  ,  # Fracción de características utilizadas para ajustar cada árbol
   # 'xgboost__min_child_weight': [1, 5, 10]   ,  # Mínima suma de pesos de la hoja
   # 'xgboost__gamma': [0, 0.1, 0.2]   ,  # Parámetro de regularización para controlar la complejidad del árbol
   # 'xgboost__reg_alpha': [0, 0.1, 0.5]   ,  # Término de regularización L1 en los pesos de las hojas
   # 'xgboost__reg_lambda': [0, 0.1, 0.5]  # Término de regularización L2 en los pesos de las hojas
#}

**xgboost__max_depth**: Este hiperparámetro especifica la profundidad máxima de cada árbol en el modelo XGBoost. Un valor más alto de max_depth permite que cada árbol sea más complejo y potencialmente capture relaciones más intrincadas en los datos. Sin embargo, un valor muy alto puede conducir al sobreajuste. En tu lista, estás probando con valores de 2, 3, 5, 7 y 10.

**xgboost__n_estimators**: Este hiperparámetro especifica el número total de árboles en el modelo XGBoost, es decir, el número de iteraciones del proceso de ajuste. Un valor más alto de n_estimators generalmente mejora el rendimiento del modelo, pero también puede aumentar el tiempo de entrenamiento. En tu lista, estás probando con 10, 100 y 500 árboles.

**xgboost__learning_rate**:

Descripción: Tasa de aprendizaje que controla la contribución de cada árbol al ensamble. Un valor menor requiere más árboles para lograr el mismo efecto.
Valores: [0.01, 0.1, 0.2]

**xgboost__subsample**:

Descripción: Fracción de muestras utilizadas para ajustar cada árbol. Controla la aleatoriedad en la construcción de cada árbol.
Valores: [0.8, 0.9, 1.0]

**xgboost__colsample_bytree**:

Descripción: Fracción de características utilizadas para ajustar cada árbol. Controla la aleatoriedad de las columnas seleccionadas para cada árbol.
Valores: [0.8, 1.0]

**xgboost__min_child_weight**:

Descripción: Mínima suma de pesos de la hoja. Controla la cantidad mínima de muestras necesarias en cada hoja del árbol.
Valores: [1, 5, 10]

**xgboost__gamma:**

Descripción: Parámetro de regularización para controlar la complejidad del árbol. Mayor valor impone una mayor regularización.
Valores: [0, 0.1, 0.2]

**xgboost__reg_alpha**:

Descripción: Término de regularización L1 en los pesos de las hojas. Controla la regularización de los pesos de las hojas.
Valores: [0, 0.1, 0.5]

**xgboost__reg_lambda**:

Descripción: Término de regularización L2 en los pesos de las hojas. Controla la regularización de los pesos de las hojas.

In [ ]:
xg_tuned = GridSearchCV(xg_pipe, xg_grid, cv=cv, n_jobs=-1)

In [ ]:
xg_tuned.fit(X_train, y_train)

In [ ]:
xg_tuned.best_estimator_

In [ ]:
acurracy_train_XGBOOST= metrics.accuracy_score(y_train, xg_tuned.predict(X_train))
acurracy_train_XGBOOST

In [ ]:
acurracy_test_XGBOOST= metrics.accuracy_score(y_test, xg_tuned.predict(X_test))
acurracy_test_XGBOOST

In [ ]:
y_pred_proba_test =  xg_tuned.predict_proba(X_test)[::,1]  # Escogemos 1 porque nos interesa el valor de YES: guiarse de la M. Confusion
auc_test_XGBOOST = metrics.roc_auc_score(y_test, y_pred_proba_test)
auc_test_XGBOOST

In [ ]:
y_pred_proba = xg_tuned.predict_proba(X_train)[::,1]  # Escogemos 1 porque nos interesa el valor de YES: guiarse de la M. Confusion
auc_train_XGBOOST= metrics.roc_auc_score(y_train, y_pred_proba)
auc_train_XGBOOST

In [ ]:
probs = xg_tuned.predict(X_train)                            #X_train
confusion_matrix2 = pd.crosstab( y_train ,probs,)  # reales, predichos SEGUN Python
confusion_matrix2

In [ ]:
probs_test = xg_tuned.predict(X_test)                            #X_train
confusion_matrix2 = pd.crosstab( y_test ,probs_test,)  # reales, predichos SEGUN Python
confusion_matrix2

#Importancia de Variables

In [ ]:
from sklearn.inspection import permutation_importance
import multiprocessing

In [ ]:
importancia = permutation_importance(
                estimator    = xg_tuned,
                X            = X_train,
                y            = y_train,
                n_repeats    = 5,
                scoring      = 'neg_root_mean_squared_error',
                n_jobs       = multiprocessing.cpu_count() - 1,
                random_state = 123
             )

# Se almacenan los resultados (media y desviación) en un dataframe
df_importancia = pd.DataFrame(
                    {k: importancia[k] for k in ['importances_mean', 'importances_std']}
                 )
df_importancia['feature'] = X_train.columns
df_importancia.sort_values('importances_mean', ascending=False)

In [ ]:
# Gráfico
fig, ax = plt.subplots(figsize=(3.5, 14))
df_importancia = df_importancia.sort_values('importances_mean', ascending=True)
ax.barh(
    df_importancia['feature'],
    df_importancia['importances_mean'],
    xerr=df_importancia['importances_std'],
    align='center',
    alpha=0
)
ax.plot(
    df_importancia['importances_mean'],
    df_importancia['feature'],
    marker="D",
    linestyle="",
    alpha=0.8,
    color="r"
)
ax.set_title('Importancia de los predictores (train)')
ax.set_xlabel('Incremento del error tras la permutación');

#Despliegue

In [ ]:
import joblib
# Recordar:
##  xgb_fit=classifier.fit(X_train, y_train)
# Guardar el modelo
joblib.dump( xg_tuned, '/content/drive/MyDrive/Colab Notebooks/modelo_BOOSTING_tunning_cloro.joblib')

In [ ]:
# Cargar el modelo
loaded_model = joblib.load('/content/drive/MyDrive/Colab Notebooks/modelo_BOOSTING_tunning_cloro.joblib')

In [ ]:
data = dataset.drop(dataset.index[muestra.index]).reset_index(drop=True).reset_index(drop=True)
display(data.head(3))
display(data.tail(3))

In [ ]:
probs = loaded_model.predict(data)
# Create the dataframe
df = pd.DataFrame(probs)
display(df)

In [ ]:
probs2 =  loaded_model.predict_proba(data)
probs2

In [ ]:
probs2_df = pd.DataFrame(probs2, columns=["Prob_0", "Prob_1"])
probs2_df

In [ ]:
# Concatenar 'df' y 'probs_df' a lo largo de las columnas (axis=1)
result_df = pd.concat([df, probs2_df], axis=1)
result_df

In [ ]:
# Renombrar la columna '0' a 'Predicción'
result_df.rename(columns={0: 'Predicción'}, inplace=True)
result_df

In [ ]:
resultados = pd.concat([data, result_df], axis=1)
resultados

#Paquetes

In [ ]:
import pkg_resources

# Lista de bibliotecas instaladas
installed_packages = pkg_resources.working_set

# Imprimir información de versión para cada biblioteca instalada
for package in installed_packages:
    print(f"{package.project_name}=={package.version}")